<a href="https://colab.research.google.com/github/dianakorka/statistical_capacity/blob/main/WTI_capacity_index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import geopandas as gpd
import plotly.express as px
import re
!pip install adjustText
from adjustText import adjust_text

## Reading and transforming data

2 excel files needed:
* latest data (WTI availability), 3 sheets,
* and the country groupings from Household_data_availability.



In [ ]:
file_path = '/content/WTI_availability_20250804.xlsx'
dfs = pd.read_excel(file_path, sheet_name=None)

for name, df in dfs.items():
    print(f"Sheet: {name}, Shape: {df.shape}")

In [ ]:
file_path = '/content/WTI_availability_20250804.xlsx'

wti_codes = pd.read_excel(file_path, sheet_name='MaxYear_by_country-indicator', skiprows=[0])

# all we want from this data sheet is the ISO codes
wti_codes[['IsoCode', 'ShortName']].head()

In [ ]:
# number of unique countries covered in this file
wti_codes.IsoCode.nunique()

In [ ]:
file_path = '/content/WTI_availability_20250804.xlsx'

wti = pd.read_excel(file_path, sheet_name='WTIData_2015-2024')

wti = wti.iloc[:, :-2]


In [ ]:
# this is the actual data we need for calculations

wti.head()

In [ ]:
# list of unique countries covered by the data file
wti.ShortName.nunique()

In [ ]:
# list of indicators covered by the data file

wti.Code.unique()

Add in country groupings from a different excel file

In [ ]:
file_path = '/content/Household_data_availability_Apr2025.xlsx'
wti_countries = pd.read_excel(file_path, sheet_name='Country definition')
wti_countries[['ShortName', 'LDC', 'LLDC', 'SIDS', 'WB_Income', 'OECD_member']].head()

In [ ]:
# we need to do some regrouping of countries
def assign_group(row):
    if row['OECD_member'] == 1:
        return 'Developed-OECD'
    elif (row['SIDS'] == 'SIDS') or (row['LDC'] == 'LDC'):
    #or (row['lldc'] == 'LLDC'):
        return 'SIDS + LDC'
    else:
        return 'Other developing and transition'

wti_countries['new_group'] = wti_countries.apply(assign_group, axis=1)

wti_countries[['ShortName', 'LDC', 'LLDC', 'SIDS', 'WB_Income', 'OECD_member', 'new_group', 'CountryType']].head()

In [ ]:
# so we get an idea how many countries are counted in which group
wti_countries.new_group.value_counts()

In [ ]:
wti_countries.CountryType.value_counts()

## Grouping for WTI indicators

The next part gives us WTI indicators, their names and we construct their categories

In [ ]:
# get definitions of indicator
file_path = '/content/WTI_availability_20250804.xlsx'
def_wti = pd.read_excel(file_path, sheet_name='ByIndicator', skiprows=[0])
def_wti.dropna(subset=['Code'], inplace=True)

In [ ]:
# drop prices, we can't use this data as it is often not produced by the NSS but rather inhouse
def_wti = def_wti.loc[def_wti['Category'] != 'ICT Prices']

In [ ]:
codes_to_keep= wti.Code.unique()

In [ ]:
# here Category is the old pre-existent  WTI category already used previously for other pursposes (by technology)
def_wti = def_wti.loc[def_wti.Code.isin(codes_to_keep)][['Code', 'Category', 'Code description']]

In [ ]:
def_wti.Category.value_counts()

In [ ]:

def_wti.sort_values(by='Code', inplace=True)
def_wti.reset_index(drop=True, inplace=True)

In [ ]:
def_wti.head()

Here I make the WTI_group categories which I will use later for more aggregate reporting

In [ ]:
codes_A = ["i112", "i271", "i271mw", "i4213tfbb"]
codes_B = ["i271G", "i271G5_pop", "i271GA"]
codes_C = ["i135tfb", "i136mwi"]
codes_D = ["i4214l", "i4214u"]
codes_E = ["i4213_256to2", "i4213_2to10", "i4213_G10"]

def_wti["wti_group"] = np.select(
    [
        def_wti["Code"].isin(codes_A),
        def_wti["Code"].isin(codes_B),
        def_wti["Code"].isin(codes_C),
        def_wti["Code"].isin(codes_D),
        def_wti["Code"].isin(codes_E)
    ],
    [
        "A. Contracting_records",
        "B. Cell_tower_coverage",
        "C. Operator_traffic",
        "D. Operators_wholesale",
        "E. Contracting_fixed_broadband_speed"
    ],
    default="Other"
)

In [ ]:
# here is the indicator grouping
def_wti[["Code", "Code description", "wti_group"]].sort_values(by="wti_group")

In [ ]:
# adding the new grouping to the existing data file
wti = wti.merge(def_wti[["Code", "Code description", "wti_group"]], on="Code", how="left")

In [ ]:
wti['DataYear'] = wti['DataYear'].astype('Int64')

In [ ]:
wti.shape

In [ ]:
# add the iso code
wti = wti.merge(wti_codes[['IsoCode', 'ShortName']], on="ShortName", how="left")

In [ ]:
wti.shape

In [ ]:
# add other country groups
wti = wti.merge(wti_countries[['ShortName', 'LDC', 'LLDC', 'SIDS', 'WB_Income', 'OECD_member', 'new_group', 'CountryType']], on="ShortName", how="left")
wti.shape

In [ ]:
# Example of data we have by country
wti[wti.ShortName == 'Mali'].head()

## Barcode plot for the country profile

Detailed plot for ONE COUNTRY ONLY: data points per year and indicator.

In [ ]:
# prepping my data to draw a barcode plot

years = [2020, 2021, 2022, 2023, 2024]

codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA', 'i136mwi',  'i135tfb','i4214l',
       'i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112', 'A.i271', 'A.i271mw', 'A.i4213tfbb', 'B.i271G' , 'B.i271G5_pop', 'B.i271GA', 'C.i136mwi',  'C.i135tfb','D.i4214l',
       'D.i4214u', 'E.i4213_256to2', 'E.i4213_2to10', 'E.i4213_G10']

wti_country = wti[(wti.ShortName == 'Sierra Leone')].pivot(index='Code', columns='DataYear', values = 'ShortName').reindex(columns=years, index=codes).notna().astype(int).reset_index()

wti_country.Code=new_names

wti_country

In [ ]:
# Convert DataFrame to numpy array
code_array = wti_country.set_index( 'Code').T.to_numpy()

fig, ax = plt.subplots(figsize=(code_array.shape[1] * 0.5, code_array.shape[0] * 0.5))

# Show barcode
ax.imshow(code_array, cmap='binary', aspect='auto', interpolation='nearest')

# Set row labels (y-axis)
ax.set_yticks(np.arange(code_array.shape[0]))
ax.set_yticklabels(wti_country.set_index( 'Code').T.index)

# Set column labels (x-axis)
ax.set_xticks(np.arange(code_array.shape[1]))
ax.set_xticklabels(wti_country.set_index( 'Code').T.columns, rotation=90)

ax.set_title('Sierra Leone supply side data availability by year and indicator')

# Optional: remove grid / frame
ax.set_xticks(np.arange(-.5, code_array.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, code_array.shape[0], 1), minor=True)
ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
ax.tick_params(which='minor', bottom=False, left=False)

plt.show()

In [ ]:
# prepping my data to draw a barcode plot

years = [2020, 2021, 2022, 2023, 2024]

codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA', 'i136mwi',  'i135tfb','i4214l',
       'i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112', 'A.i271', 'A.i271mw', 'A.i4213tfbb', 'B.i271G' , 'B.i271G5_pop', 'B.i271GA', 'C.i136mwi',  'C.i135tfb','D.i4214l',
       'D.i4214u', 'E.i4213_256to2', 'E.i4213_2to10', 'E.i4213_G10']

wti_country = wti[(wti.ShortName == 'Mali')].pivot(index='Code', columns='DataYear', values = 'ShortName').reindex(columns=years, index=codes).notna().astype(int).reset_index()

wti_country.Code=new_names

# Convert DataFrame to numpy array
code_array = wti_country.set_index( 'Code').T.to_numpy()

fig, ax = plt.subplots(figsize=(code_array.shape[1] * 0.5, code_array.shape[0] * 0.5))

# Show barcode
ax.imshow(code_array, cmap='binary', aspect='auto', interpolation='nearest')

ax.set_title('Mali supply side data availability by year and indicator')

# Set row labels (y-axis)
ax.set_yticks(np.arange(code_array.shape[0]))
ax.set_yticklabels(wti_country.set_index( 'Code').T.index)

# Set column labels (x-axis)
ax.set_xticks(np.arange(code_array.shape[1]))
ax.set_xticklabels(wti_country.set_index( 'Code').T.columns, rotation=90)

# Optional: remove grid / frame
ax.set_xticks(np.arange(-.5, code_array.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, code_array.shape[0], 1), minor=True)
ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
ax.tick_params(which='minor', bottom=False, left=False)

plt.show()

## Preparing data for calculating average availability by country-indicator

data_in_past_5_years = number of years data was available in the part 5 years for country I, indicator J. Number of times data was available in the past 5 years (2020-2024) for country I, indicator J.

5y_percent and 10y_percent = proportion years when data was available in the past 5y or 10y (2015-2024) for country I, indicator J. For example 5 years out of 5=100%, 5 years out of 10=50%.

NEED TO GROUP THE DATA BY COUNTRY-INDICATOR PAIR

In [ ]:
# shows reference years initially included in the dataset

wti.DataYear.describe()

In [ ]:
# keep only 2020-2024 time frame for analysis

wti = wti[wti["DataYear"] >= 2020].reset_index(drop=True)


In [ ]:
wti.DataYear.describe()

In [ ]:
# Count the number of rows per Country + Indicator
wti_counts = wti.groupby(["ShortName", "Code", "RegionName", "Code description", "wti_group", "IsoCode", "new_group", "CountryType", "WB_Income"]).size().reset_index(name="5Y")


In [ ]:
wti_counts.head()

In [ ]:
wti_counts['5Y'].isna().sum()

In [ ]:
# this shows there is no missing data here

wti_counts['5Y'].describe()

In [ ]:
wti_counts.IsoCode.nunique()

In [ ]:
wti_counts['5Y_p'] = wti_counts['5Y'] / 5 *100

In [ ]:
# example of aggregated data output for one country
wti_counts[wti_counts.ShortName== 'Mali']

## DOT PLOT number of countries reporting data by country group


Count how many countries have data by indicator and by country group. We take the 5 years averages calculated above.

In [ ]:
# sanity check for one region: CIS countries, smallest country group
# shows there are 7 CIS countries that reported data for indicator i4214u, as in the dotplot

wti_counts[(wti_counts.RegionName=='CIS countries') & (wti_counts.Code=='i4214u')]

In [ ]:
# these are the calculations for all the regions
wti_counts[wti_counts.RegionName != 'Other Economies'].groupby(['Code', 'RegionName'])['ShortName'].nunique().reset_index(name='num_countries')


In [ ]:
# region totals
wti_counts.groupby(['RegionName'])['ShortName'].nunique()

In [ ]:
import plotly.express as px


# Define custom order and colors
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'   # Orange
]

# Create a color mapping dict
color_map = dict(zip(custom_order, custom_colors))

# Your custom order and new names
codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA',
         'i136mwi',  'i135tfb','i4214l','i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112', 'A.i271', 'A.i271mw', 'A.i4213tfbb', 'B.i271G' , 'B.i271G5_pop',
             'B.i271GA', 'C.i136mwi',  'C.i135tfb','D.i4214l','D.i4214u',
             'E.i4213_256to2', 'E.i4213_2to10', 'E.i4213_G10']

# Create a mapping dict from old code to new names
rename_map = dict(zip(codes, new_names))

# Calculate the total unique countries per region
region_totals = wti_counts.groupby(['RegionName'])['ShortName'].nunique()


df = wti_counts[wti_counts.RegionName != 'Other Economies'].groupby(['Code', 'RegionName'])['ShortName'].nunique().reset_index(name='num_countries').set_index('Code').loc[codes].reset_index()
df['Code'] = df['Code'].map(rename_map)

# Map region totals and compute relative_countries
df['relative_countries'] = df['num_countries'] / df['RegionName'].map(region_totals)

# Create the scatter plot with custom order and colors
fig = px.scatter(
    df,
    y="Code",
    x="relative_countries",
    color="RegionName",
    symbol="RegionName",
    category_orders={"RegionName": custom_order},  # enforce order
    color_discrete_map=color_map                   # enforce colors
)

# Optional: update marker size
fig.update_traces(marker_size=15)

# Remove gray background and grid lines
fig.update_layout(
    xaxis_title="Proportion of Countries",
    yaxis_title=" ",
    plot_bgcolor='#F5F5F5',  # white background
    paper_bgcolor='white', # white surrounding background
    xaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),  # remove x-axis grid lines
    yaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),   # remove y-axis grid lines
    legend=dict(font=dict(size=18))
)

fig.show()


In [ ]:
import plotly.express as px


# Define custom order and colors
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'   # Orange
]

# Create a color mapping dict
color_map = dict(zip(custom_order, custom_colors))

# Your custom order and new names
codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA',
         'i136mwi',  'i135tfb','i4214l','i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112', 'A.i271', 'A.i271mw', 'A.i4213tfbb', 'B.i271G' , 'B.i271G5_pop',
             'B.i271GA', 'C.i136mwi',  'C.i135tfb','D.i4214l','D.i4214u',
             'E.i4213_256to2', 'E.i4213_2to10', 'E.i4213_G10']

# Create a mapping dict from old code to new names
rename_map = dict(zip(codes, new_names))


df = wti_counts[wti_counts.RegionName != 'Other Economies'].groupby(['Code', 'RegionName'])['ShortName'].nunique().reset_index(name='num_countries').set_index('Code').loc[codes].reset_index()
df['Code'] = df['Code'].map(rename_map)


# Create the scatter plot with custom order and colors
fig = px.scatter(
    df,
    y="Code",
    x="num_countries",
    color="RegionName",
    symbol="RegionName",
    category_orders={"RegionName": custom_order},  # enforce order
    color_discrete_map=color_map                   # enforce colors
)

# Optional: update marker size
fig.update_traces(marker_size=15)

# Remove gray background and grid lines
fig.update_layout(
    xaxis_title="Number of Countries",
    yaxis_title=" ",
    plot_bgcolor='#F5F5F5',  # white background
    paper_bgcolor='white', # white surrounding background
    xaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),  # remove x-axis grid lines
    yaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),   # remove y-axis grid lines
    legend=dict(font=dict(size=18))
)

fig.show()


In [ ]:

# Define your custom order
custom_order = ['Developed-OECD', 'Other developing and transition', 'SIDS + LDC']


# Your custom order and new names
codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA',
         'i136mwi',  'i135tfb','i4214l','i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112', 'A.i271', 'A.i271mw', 'A.i4213tfbb', 'B.i271G' , 'B.i271G5_pop',
             'B.i271GA', 'C.i136mwi',  'C.i135tfb','D.i4214l','D.i4214u',
             'E.i4213_256to2', 'E.i4213_2to10', 'E.i4213_G10']

# Create a mapping dict from old code to new names
rename_map = dict(zip(codes, new_names))

# Calculate the total unique countries per group
totals = wti_counts.groupby(['new_group'])['ShortName'].nunique()

df = wti_counts.groupby(['Code', 'new_group'])['ShortName'].nunique().reset_index(name='num_countries').set_index('Code').loc[codes].reset_index()
df['Code'] = df['Code'].map(rename_map)

# Map group totals and compute relative_countries
df['relative_countries'] = df['num_countries'] / df['new_group'].map(totals)

# Create the scatter plot with custom order and colors
fig = px.scatter(
    df,
    y="Code",
    x="relative_countries",
    color="new_group",
    symbol="new_group",
    category_orders={"new_group": custom_order}  # enforce order
)

# Optional: update marker size
fig.update_traces(marker_size=15)

# Remove gray background and grid lines
fig.update_layout(
    xaxis_title="Proportion of Countries",
    yaxis_title=" ",
    plot_bgcolor='#F5F5F5',  # white background
    paper_bgcolor='white', # white surrounding background
    xaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),  # remove x-axis grid lines
    yaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),   # remove y-axis grid lines
    legend=dict(font=dict(size=18))
)

fig.show()


In [ ]:

# Define your custom order
custom_order = ['Developed-OECD', 'Other developing and transition', 'SIDS + LDC']


# Your custom order and new names
codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA',
         'i136mwi',  'i135tfb','i4214l','i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112', 'A.i271', 'A.i271mw', 'A.i4213tfbb', 'B.i271G' , 'B.i271G5_pop',
             'B.i271GA', 'C.i136mwi',  'C.i135tfb','D.i4214l','D.i4214u',
             'E.i4213_256to2', 'E.i4213_2to10', 'E.i4213_G10']

# Create a mapping dict from old code to new names
rename_map = dict(zip(codes, new_names))

df = wti_counts.groupby(['Code', 'new_group'])['ShortName'].nunique().reset_index(name='num_countries').set_index('Code').loc[codes].reset_index()
df['Code'] = df['Code'].map(rename_map)

# Create the scatter plot with custom order and colors
fig = px.scatter(
    df,
    y="Code",
    x="num_countries",
    color="new_group",
    symbol="new_group",
    category_orders={"new_group": custom_order}  # enforce order
)

# Optional: update marker size
fig.update_traces(marker_size=15)

# Remove gray background and grid lines
fig.update_layout(
    xaxis_title="Number of Countries",
    yaxis_title=" ",
    plot_bgcolor='#F5F5F5',  # white background
    paper_bgcolor='white', # white surrounding background
    xaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),  # remove x-axis grid lines
    yaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),   # remove y-axis grid lines
    legend=dict(font=dict(size=18))
)

fig.show()


In [ ]:

# Define your custom order
custom_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']


# Your custom order and new names
codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA',
         'i136mwi',  'i135tfb','i4214l','i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112', 'A.i271', 'A.i271mw', 'A.i4213tfbb', 'B.i271G' , 'B.i271G5_pop',
             'B.i271GA', 'C.i136mwi',  'C.i135tfb','D.i4214l','D.i4214u',
             'E.i4213_256to2', 'E.i4213_2to10', 'E.i4213_G10']

# Create a mapping dict from old code to new names
rename_map = dict(zip(codes, new_names))

# Calculate the total unique countries per group
totals = wti_counts.groupby(['WB_Income'])['ShortName'].nunique()

df = wti_counts.groupby(['Code', 'WB_Income'])['ShortName'].nunique().reset_index(name='num_countries').set_index('Code').loc[codes].reset_index()
df['Code'] = df['Code'].map(rename_map)

# Map group totals and compute relative_countries
df['relative_countries'] = df['num_countries'] / df['WB_Income'].map(totals)

# Create the scatter plot with custom order and colors
fig = px.scatter(
    df,
    y="Code",
    x="relative_countries",
    color="WB_Income",
    symbol="WB_Income",
    category_orders={"WB_Income": custom_order}  # enforce order
)

# Optional: update marker size
fig.update_traces(marker_size=15)

# Remove gray background and grid lines
fig.update_layout(
    xaxis_title="Proportion of Countries",
    yaxis_title=" ",
    plot_bgcolor='#F5F5F5',  # white background
    paper_bgcolor='white', # white surrounding background
    xaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),  # remove x-axis grid lines
    yaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),   # remove y-axis grid lines
    legend=dict(font=dict(size=18))
)

fig.show()


In [ ]:

# Define your custom order
custom_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']


# Your custom order and new names
codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA',
         'i136mwi',  'i135tfb','i4214l','i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112', 'A.i271', 'A.i271mw', 'A.i4213tfbb', 'B.i271G' , 'B.i271G5_pop',
             'B.i271GA', 'C.i136mwi',  'C.i135tfb','D.i4214l','D.i4214u',
             'E.i4213_256to2', 'E.i4213_2to10', 'E.i4213_G10']

# Create a mapping dict from old code to new names
rename_map = dict(zip(codes, new_names))

df = wti_counts.groupby(['Code', 'WB_Income'])['ShortName'].nunique().reset_index(name='num_countries').set_index('Code').loc[codes].reset_index()
df['Code'] = df['Code'].map(rename_map)

# Create the scatter plot with custom order and colors
fig = px.scatter(
    df,
    y="Code",
    x="num_countries",
    color="WB_Income",
    symbol="WB_Income",
    category_orders={"WB_Income": custom_order}  # enforce order
)

# Optional: update marker size
fig.update_traces(marker_size=15)

# Remove gray background and grid lines
fig.update_layout(
    xaxis_title="Number of Countries",
    yaxis_title=" ",
    plot_bgcolor='#F5F5F5',  # white background
    paper_bgcolor='white', # white surrounding background
    xaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),  # remove x-axis grid lines
    yaxis=dict(showgrid=False, title_font=dict(size=18), tickfont=dict(size=18)),   # remove y-axis grid lines
    legend=dict(font=dict(size=18))
)

fig.show()


## Availability charts (by country group)

Pivot back my data so I can plot it


Plotting everything without any assumptions about missing data.


In [ ]:
pivoted = wti_counts.pivot(index=['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType', 'WB_Income'], columns='Code', values=['5Y', '5Y_p'])
# Replace NaNs with 0 -- OK, no longer filling NAs with 0, so no assumptions about missing data
# pivoted = pivoted.fillna(0)

# Flatten and rename the columns
pivoted.columns = [
    f"{indicator}_{suffix}"
    for suffix, indicator in pivoted.columns
]

pivoted.reset_index(inplace=True)

pivoted.head()

In [ ]:
pivoted.ShortName.nunique()

In [ ]:
# add an average of all these detailed _5Y indicators so we'll be able to plot against a similar average for timeliness

pivoted['availability_5Y'] = pivoted[['i112_5Y', 'i135tfb_5Y', 'i136mwi_5Y', 'i271_5Y',
       'i271G_5Y', 'i271G5_pop_5Y', 'i271GA_5Y', 'i271mw_5Y',
       'i4213_256to2_5Y', 'i4213_2to10_5Y', 'i4213_G10_5Y', 'i4213tfbb_5Y',
       'i4214l_5Y', 'i4214u_5Y']].mean(axis=1)

pivoted['availability_5Y'].describe()

Basically for missing data no assumption is made. So we just take the average of the existing data.

In [ ]:
# this code shows what happends with the missing data
pivoted[pivoted['i271G5_pop_5Y'].isna()==True].head()

In [ ]:
# this is the list of regions present in the data
pivoted.RegionName.unique()

In [ ]:
pivoted.columns

In [ ]:
## this dataset contains imputed 0s as in 0 data point available
pivoted.i4213_2to10_5Y.describe()

In [ ]:
pivoted.i4213_2to10_5Y.isna().sum()

In [ ]:
# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    #'#696969'
    #'#999999',   # Grey
    ]

rename_dict = {
 'i112_5Y': 'A i112',
 'i135tfb_5Y': 'C i135tfb',
 'i136mwi_5Y': 'C i136mwi',
 'i271_5Y': 'A i271',
 'i271G_5Y': 'B i271G',
 'i271G5_pop_5Y': 'B i271G5_pop',
 'i271GA_5Y': 'B i271GA',
 'i271mw_5Y': 'A i271mw',
 'i4213_256to2_5Y': 'E i4213_256to2',
 'i4213_2to10_5Y': 'E i4213_2to10',
 'i4213_G10_5Y': 'E i4213_G10',
 'i4213tfbb_5Y': 'A i4213tfbb',
 'i4214l_5Y': 'D i4214l',
 'i4214u_5Y': 'D i4214u'
}

pivoted[['RegionName','i112_5Y', 'i135tfb_5Y',
       'i136mwi_5Y', 'i271_5Y', 'i271G_5Y', 'i271G5_pop_5Y', 'i271GA_5Y',
       'i271mw_5Y', 'i4213_256to2_5Y', 'i4213_2to10_5Y', 'i4213_G10_5Y',
       'i4213tfbb_5Y', 'i4214l_5Y', 'i4214u_5Y']].rename(
     columns=rename_dict).groupby('RegionName').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Europe countries', ascending=False).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  color=custom_colors,
                                                  width=0.7)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
pivoted.WB_Income.unique()

In [ ]:

rename_dict = {
  'i112_5Y': 'A i112',
 'i135tfb_5Y': 'C i135tfb',
 'i136mwi_5Y': 'C i136mwi',
 'i271_5Y': 'A i271',
 'i271G_5Y': 'B i271G',
 'i271G5_pop_5Y': 'B i271G5_pop',
 'i271GA_5Y': 'B i271GA',
 'i271mw_5Y': 'A i271mw',
 'i4213_256to2_5Y': 'E i4213_256to2',
 'i4213_2to10_5Y': 'E i4213_2to10',
 'i4213_G10_5Y': 'E i4213_G10',
 'i4213tfbb_5Y': 'A i4213tfbb',
 'i4214l_5Y': 'D i4214l',
 'i4214u_5Y': 'D i4214u'
}

# Define your custom order
custom_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']

pivoted[['WB_Income','i112_5Y', 'i135tfb_5Y',
       'i136mwi_5Y', 'i271_5Y', 'i271G_5Y', 'i271G5_pop_5Y', 'i271GA_5Y',
       'i271mw_5Y', 'i4213_256to2_5Y', 'i4213_2to10_5Y', 'i4213_G10_5Y',
       'i4213tfbb_5Y', 'i4214l_5Y', 'i4214u_5Y'
 ]].rename(columns=rename_dict).groupby('WB_Income').mean(numeric_only=True).T.sort_values(
     by='Upper middle income', ascending=False)[custom_order].plot(kind='bar',
                                                figsize=(12, 6),
                                                width=0.6)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1) # places legend bottom

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.3)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=4,
    fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
pivoted.new_group.unique()

In [ ]:

rename_dict = {
  'i112_5Y': 'A i112',
 'i135tfb_5Y': 'C i135tfb',
 'i136mwi_5Y': 'C i136mwi',
 'i271_5Y': 'A i271',
 'i271G_5Y': 'B i271G',
 'i271G5_pop_5Y': 'B i271G5_pop',
 'i271GA_5Y': 'B i271GA',
 'i271mw_5Y': 'A i271mw',
 'i4213_256to2_5Y': 'E i4213_256to2',
 'i4213_2to10_5Y': 'E i4213_2to10',
 'i4213_G10_5Y': 'E i4213_G10',
 'i4213tfbb_5Y': 'A i4213tfbb',
 'i4214l_5Y': 'D i4214l',
 'i4214u_5Y': 'D i4214u'
}

# Define your custom order
custom_order = ['Developed-OECD', 'Other developing and transition', 'SIDS + LDC']

pivoted[['new_group','i112_5Y', 'i135tfb_5Y',
       'i136mwi_5Y', 'i271_5Y', 'i271G_5Y', 'i271G5_pop_5Y', 'i271GA_5Y',
       'i271mw_5Y', 'i4213_256to2_5Y', 'i4213_2to10_5Y', 'i4213_G10_5Y',
       'i4213tfbb_5Y', 'i4214l_5Y', 'i4214u_5Y'
 ]].rename(columns=rename_dict).groupby('new_group').mean(numeric_only=True).T.sort_values(
     by='Other developing and transition', ascending=False)[custom_order].plot(kind='bar',
                                                figsize=(12, 6),
                                                width=0.6)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1) # places legend bottom

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.3)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=3,
    fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
pivoted.CountryType.unique()

In [ ]:

rename_dict = {
  'i112_5Y': 'A i112',
 'i135tfb_5Y': 'C i135tfb',
 'i136mwi_5Y': 'C i136mwi',
 'i271_5Y': 'A i271',
 'i271G_5Y': 'B i271G',
 'i271G5_pop_5Y': 'B i271G5_pop',
 'i271GA_5Y': 'B i271GA',
 'i271mw_5Y': 'A i271mw',
 'i4213_256to2_5Y': 'E i4213_256to2',
 'i4213_2to10_5Y': 'E i4213_2to10',
 'i4213_G10_5Y': 'E i4213_G10',
 'i4213tfbb_5Y': 'A i4213tfbb',
 'i4214l_5Y': 'D i4214l',
 'i4214u_5Y': 'D i4214u'
}

# Define your custom order
custom_order = [ 'Developed', 'Developing']

pivoted[['CountryType','i112_5Y', 'i135tfb_5Y',
       'i136mwi_5Y', 'i271_5Y', 'i271G_5Y', 'i271G5_pop_5Y', 'i271GA_5Y',
       'i271mw_5Y', 'i4213_256to2_5Y', 'i4213_2to10_5Y', 'i4213_G10_5Y',
       'i4213tfbb_5Y', 'i4214l_5Y', 'i4214u_5Y'
 ]].rename(columns=rename_dict).groupby('CountryType').mean(numeric_only=True).T.sort_values(
     by='Developing', ascending=False)[custom_order].plot(kind='bar',
                                                figsize=(12, 6),
                                                width=0.6)
plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1) # places legend bottom

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.3)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=2,
    fontsize=14
)
plt.tight_layout()
plt.show()

## Charts for country profile (one country)

In [ ]:
pivoted[pivoted.ShortName=='Madagascar'][['RegionName','i112_5Y', 'i135tfb_5Y',
       'i136mwi_5Y', 'i271_5Y', 'i271G_5Y', 'i271G5_pop_5Y', 'i271GA_5Y',
       'i271mw_5Y', 'i4213_256to2_5Y', 'i4213_2to10_5Y', 'i4213_G10_5Y',
       'i4213tfbb_5Y', 'i4214l_5Y', 'i4214u_5Y']]

In [ ]:
# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    #'#696969'
    #'#999999',   # Grey
    ]

rename_dict = {
 'i112_5Y': 'A i112',
 'i135tfb_5Y': 'C i135tfb',
 'i136mwi_5Y': 'C i136mwi',
 'i271_5Y': 'A i271',
 'i271G_5Y': 'B i271G',
 'i271G5_pop_5Y': 'B i271G5_pop',
 'i271GA_5Y': 'B i271GA',
 'i271mw_5Y': 'A i271mw',
 'i4213_256to2_5Y': 'E i4213_256to2',
 'i4213_2to10_5Y': 'E i4213_2to10',
 'i4213_G10_5Y': 'E i4213_G10',
 'i4213tfbb_5Y': 'A i4213tfbb',
 'i4214l_5Y': 'D i4214l',
 'i4214u_5Y': 'D i4214u'
}

pivoted[pivoted.ShortName=='Madagascar'][['RegionName','i112_5Y', 'i135tfb_5Y',
       'i136mwi_5Y', 'i271_5Y', 'i271G_5Y', 'i271G5_pop_5Y', 'i271GA_5Y',
       'i271mw_5Y', 'i4213_256to2_5Y', 'i4213_2to10_5Y', 'i4213_G10_5Y',
       'i4213tfbb_5Y', 'i4214l_5Y', 'i4214u_5Y']].rename(
     columns=rename_dict).groupby('RegionName').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Africa', ascending=False).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  color=custom_colors,
                                                  width=0.7)
#plt.title('MADAGASCAR: Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# Step 1: Get current handles and labels
#handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
#label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
#plt.legend(
    #[label_to_handle[label] for label in custom_order],
   # custom_order,
    #loc='upper center',
    #bbox_to_anchor=(0.5, -0.5),
    #ncol=7,
   # fontsize=12
#)
plt.legend().remove()


plt.tight_layout()
plt.show()

## Group-wise calculations for availability (by group of indicators)


Within the group take the best availability - the NSS has the capacity to produce this data, but maybe it chooses not to produce all indicators because lack of policy interest

In [ ]:
# Make calculations for the group of indicators - take the maximum per group

wti_counts.groupby(["ShortName", "RegionName", "wti_group", "IsoCode", 'new_group', 'CountryType', 'WB_Income'], as_index=False)["5Y"].max().head()

In [ ]:
wti_counts_group = wti_counts.groupby(["ShortName", "RegionName", "wti_group", "IsoCode", 'new_group', 'CountryType', 'WB_Income'], as_index=False)["5Y"].max()

# add percentage column

wti_counts_group['5Y_p'] = wti_counts_group['5Y'] / 5 *100

pivoted_group = wti_counts_group.pivot(index=['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType', 'WB_Income'], columns='wti_group', values=['5Y', '5Y_p'])

# Pivot back my data so I ca plot it
# Replace NaNs with 0 - rolling this back, I' no longer filling NAs with 0
#pivoted_group = pivoted_group.fillna(0)

# Flatten and rename the columns
pivoted_group.columns = [
    f"{indicator}_{suffix}"
    for suffix, indicator in pivoted_group.columns
]

pivoted_group.reset_index(inplace=True)

pivoted_group.head()

In [ ]:
pivoted_group.IsoCode.nunique()

In [ ]:
pivoted_group['E. Contracting_fixed_broadband_speed_5Y'].describe()

In [ ]:
pivoted_group.columns

In [ ]:
pivoted_group.RegionName.unique()

In [ ]:
pivoted_group[['A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
       'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
       'E. Contracting_fixed_broadband_speed_5Y']].describe()

In [ ]:
# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    #'#696969'
    #'#999999',   # Grey
    ]

rename_dict = {
'B. Cell_tower_coverage_5Y': "B Cell tower coverage",
'A. Contracting_records_5Y':"A Contracting records",
'C. Operator_traffic_5Y':"C Operator traffic",
'D. Operators_wholesale_5Y':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_5Y': "E Contracting fixed-broadband speed"
}

pivoted_group[['RegionName','B. Cell_tower_coverage_5Y',
       'A. Contracting_records_5Y', 'C. Operator_traffic_5Y',
       'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']].rename(
     columns=rename_dict).groupby('RegionName').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Europe countries', ascending=False).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  color=custom_colors,
                                                  width=0.7)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.6)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.1),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    ]

rename_dict = {
'B. Cell_tower_coverage_5Y': "B Cell tower coverage",
'A. Contracting_records_5Y':"A Contracting records",
'C. Operator_traffic_5Y':"C Operator traffic",
'D. Operators_wholesale_5Y':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_5Y': "E Contracting fixed-broadband speed"
}


subset = pivoted_group[['RegionName','B. Cell_tower_coverage_5Y',
       'A. Contracting_records_5Y', 'C. Operator_traffic_5Y',
       'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']].rename(
     columns=rename_dict)

# Calculate means and counts
means = subset.groupby('RegionName').mean(numeric_only=True).T
counts = subset.groupby('RegionName').count().T

# Sort by Europe countries
means_sorted = means.sort_values(by='Europe countries', ascending=False)[custom_order]
counts_sorted = counts.loc[means_sorted.index][custom_order]

# Set width factor range
min_width_factor = 0.3
max_width_factor = 1.0

# Create figure
fig, ax = plt.subplots(figsize=(18, 6))

# Plot each region separately to control width
x = np.arange(len(means_sorted.index))
base_width = 0.6 / len(custom_order)

for i, region in enumerate(custom_order):
    # Calculate width for each bar WITHIN THIS GROUP
    group_max_count = counts_sorted[region].max()

    # Normalize within the group
    width_factors = counts_sorted[region] / group_max_count * (max_width_factor - min_width_factor) + min_width_factor

    # Plot bars with individual widths
    for j, (indicator, value) in enumerate(means_sorted[region].items()):
        bar_width = base_width * width_factors.iloc[j]
        bar_x = x[j] + i * base_width - (len(custom_order) - 1) * base_width / 2

        ax.bar(
            bar_x,
            value,
            bar_width,
            color=custom_colors[i]
        )

        # Add text label on top of bar
        count = int(counts_sorted[region].iloc[j])
        if count > 0:
            ax.text(
                bar_x,
                value,
                f'n={count}',
                ha='center',
                va='bottom',
                fontsize=9,
                rotation=0
            )

ax.set_xlabel('', fontsize=14)
ax.set_ylabel('number of data points', fontsize=14)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
ax.set_xticks(x)
ax.set_xticklabels(means_sorted.index, rotation=45, fontsize=14, ha='right')
ax.tick_params(axis='y', labelsize=14)

# Create custom legend
legend_handles = [Patch(facecolor=custom_colors[i], label=region)
                  for i, region in enumerate(custom_order)] #

plt.subplots_adjust(bottom=0.7)
plt.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.7),
    ncol=1,
    fontsize=14
)

plt.tight_layout()
plt.show()



In [ ]:
rename_dict = {
'B. Cell_tower_coverage_5Y': "B Cell tower coverage",
'A. Contracting_records_5Y':"A Contracting records",
'C. Operator_traffic_5Y':"C Operator traffic",
'D. Operators_wholesale_5Y':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_5Y': "E Contracting fixed-broadband speed"
}

# Define your custom order
custom_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']

pivoted_group[['WB_Income','B. Cell_tower_coverage_5Y',
       'A. Contracting_records_5Y', 'C. Operator_traffic_5Y',
       'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y'
 ]].rename(columns=rename_dict).groupby('WB_Income').mean(numeric_only=True).T.sort_values(
     by='Upper middle income', ascending=False)[custom_order].plot(kind='bar',
                                                figsize=(12, 6),
                                                width=0.6)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1) # places legend bottom

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.7)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.1),
    ncol=4,
    fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Define your custom order
custom_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']
#, 'Other Economies']

rename_dict = {
'B. Cell_tower_coverage_5Y': "B Cell tower coverage",
'A. Contracting_records_5Y':"A Contracting records",
'C. Operator_traffic_5Y':"C Operator traffic",
'D. Operators_wholesale_5Y':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_5Y': "E Contracting fixed-broadband speed"
}

custom_colors = ['#1f77b4',  # Blue
'#ff7f0e' , # Orange
'#2ca02c',  # Green
'#d62728'  # Red
]

subset = pivoted_group[['WB_Income','B. Cell_tower_coverage_5Y',
       'A. Contracting_records_5Y', 'C. Operator_traffic_5Y',
       'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']].rename(
     columns=rename_dict)

# Calculate means and counts
means = subset.groupby('WB_Income').mean(numeric_only=True).T
counts = subset.groupby('WB_Income').count().T

# Sort by Europe countries
means_sorted = means.sort_values(by='Upper middle income', ascending=False)[custom_order]
counts_sorted = counts.loc[means_sorted.index][custom_order]

# Set width factor range
min_width_factor = 0.3
max_width_factor = 1.0

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

# Plot each region separately to control width
x = np.arange(len(means_sorted.index))
base_width = 0.6 / len(custom_order)

for i, region in enumerate(custom_order):
    # Calculate width for each bar WITHIN THIS GROUP
    group_max_count = counts_sorted[region].max()

    # Normalize within the group
    width_factors = counts_sorted[region] / group_max_count * (max_width_factor - min_width_factor) + min_width_factor

    # Plot bars with individual widths
    for j, (indicator, value) in enumerate(means_sorted[region].items()):
        bar_width = base_width * width_factors.iloc[j]
        bar_x = x[j] + i * base_width - (len(custom_order) - 1) * base_width / 2

        ax.bar(
            bar_x,
            value,
            bar_width,
            color=custom_colors[i]
        )

        # Add text label on top of bar
        count = int(counts_sorted[region].iloc[j])
        if count > 0:
            ax.text(
                bar_x,
                value,
                f'n={count}',
                ha='center',
                va='bottom',
                fontsize=8,
                rotation=0
            )

ax.set_xlabel('', fontsize=14)
ax.set_ylabel('number of data points', fontsize=14)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
ax.set_xticks(x)
ax.set_xticklabels(means_sorted.index, rotation=45, fontsize=14, ha='right')
ax.tick_params(axis='y', labelsize=14)

# Create custom legend
legend_handles = [Patch(facecolor=custom_colors[i], label=region)
                  for i, region in enumerate(custom_order)] #

plt.subplots_adjust(bottom=0.7)
plt.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.7),
    ncol=1,
    fontsize=14
)

plt.tight_layout()
plt.show()



In [ ]:
rename_dict = {
'B. Cell_tower_coverage_5Y': "B Cell tower coverage",
'A. Contracting_records_5Y':"A Contracting records",
'C. Operator_traffic_5Y':"C Operator traffic",
'D. Operators_wholesale_5Y':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_5Y': "E Contracting fixed-broadband speed"
}

# Define your custom order
custom_order = ['Developed-OECD', 'Other developing and transition', 'SIDS + LDC']

pivoted_group[['new_group','B. Cell_tower_coverage_5Y',
       'A. Contracting_records_5Y', 'C. Operator_traffic_5Y',
       'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y'
 ]].rename(columns=rename_dict).groupby('new_group').mean(numeric_only=True).T.sort_values(
     by='Other developing and transition', ascending=False)[custom_order].plot(kind='bar',
                                                figsize=(12, 6),
                                                width=0.6)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1) # places legend bottom

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.7)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.1),
    ncol=4,
    fontsize=14
)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Define your custom order
custom_order = ['Developed-OECD', 'Other developing and transition', 'SIDS + LDC']

rename_dict = {
'B. Cell_tower_coverage_5Y': "B Cell tower coverage",
'A. Contracting_records_5Y':"A Contracting records",
'C. Operator_traffic_5Y':"C Operator traffic",
'D. Operators_wholesale_5Y':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_5Y': "E Contracting fixed-broadband speed"
}

custom_colors = ['#1f77b4',  # Blue
'#ff7f0e' , # Orange
'#2ca02c',  # Green
'#d62728'  # Red
]

subset = pivoted_group[['new_group','B. Cell_tower_coverage_5Y',
       'A. Contracting_records_5Y', 'C. Operator_traffic_5Y',
       'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']].rename(
     columns=rename_dict)

# Calculate means and counts
means = subset.groupby('new_group').mean(numeric_only=True).T
counts = subset.groupby('new_group').count().T

# Sort by Europe countries
means_sorted = means.sort_values(by='Other developing and transition', ascending=False)[custom_order]
counts_sorted = counts.loc[means_sorted.index][custom_order]

# Set width factor range
min_width_factor = 0.3
max_width_factor = 1.0

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

# Plot each region separately to control width
x = np.arange(len(means_sorted.index))
base_width = 0.6 / len(custom_order)

for i, region in enumerate(custom_order):
    # Calculate width for each bar WITHIN THIS GROUP
    group_max_count = counts_sorted[region].max()

    # Normalize within the group
    width_factors = counts_sorted[region] / group_max_count * (max_width_factor - min_width_factor) + min_width_factor

    # Plot bars with individual widths
    for j, (indicator, value) in enumerate(means_sorted[region].items()):
        bar_width = base_width * width_factors.iloc[j]
        bar_x = x[j] + i * base_width - (len(custom_order) - 1) * base_width / 2

        ax.bar(
            bar_x,
            value,
            bar_width,
            color=custom_colors[i]
        )

        # Add text label on top of bar
        count = int(counts_sorted[region].iloc[j])
        if count > 0:
            ax.text(
                bar_x,
                value,
                f'n={count}',
                ha='center',
                va='bottom',
                fontsize=10,
                rotation=0
            )

ax.set_xlabel('', fontsize=14)
ax.set_ylabel('number of data points', fontsize=14)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
ax.set_xticks(x)
ax.set_xticklabels(means_sorted.index, rotation=45, fontsize=14, ha='right')
ax.tick_params(axis='y', labelsize=14)

# Create custom legend
legend_handles = [Patch(facecolor=custom_colors[i], label=region)
                  for i, region in enumerate(custom_order)] #

plt.subplots_adjust(bottom=0.7)
plt.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.7),
    ncol=1,
    fontsize=14
)

plt.tight_layout()
plt.show()



## Charts for country profile (one country)

In [ ]:
pivoted_group[pivoted_group.ShortName=='Madagascar'][['RegionName','B. Cell_tower_coverage_5Y',
       'A. Contracting_records_5Y', 'C. Operator_traffic_5Y',
       'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']]

In [ ]:
# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    #'#696969'
    #'#999999',   # Grey
    ]

rename_dict = {
'B. Cell_tower_coverage_5Y': "B Cell tower coverage",
'A. Contracting_records_5Y':"A Contracting records",
'C. Operator_traffic_5Y':"C Operator traffic",
'D. Operators_wholesale_5Y':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_5Y': "E Contracting fixed-broadband speed"
}

pivoted_group[pivoted_group.ShortName=='Madagascar'][['RegionName','B. Cell_tower_coverage_5Y',
       'A. Contracting_records_5Y', 'C. Operator_traffic_5Y',
       'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']].rename(
     columns=rename_dict).groupby('RegionName').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Africa', ascending=False).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  color=custom_colors,
                                                  width=0.7)
#plt.title('Average number of data points per indicator in the past 5 years (2020-2024)')
plt.xlabel('')
plt.ylabel('number of data points', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.6)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.1),
    ncol=7,
    fontsize=12
)
plt.legend().remove()
plt.tight_layout()
plt.show()

## Timeliness

We have 2020-2024 data. we will further trim it to have only the latest year available.

For each indicator take the time lag with which data is available (with reference year 2025). Example: For country I, indicator J data was last available in 2020. Timeliness is 2025 (reference year) - 2024 (last year data available for wti1 in Austria) = 1.

Here higher values are associated with a bad performance (data is less timely if it is older).


In [ ]:
# for example for Afghanistan we have a max of 4 observations per country-indicator
wti[wti.ShortName=='Afghanistan'].DataYear.unique()

In [ ]:
wti[['Code', 'ShortName', 'DataYear']][(wti.ShortName=='Afghanistan') & (wti.Code=='i271G')]

In [ ]:
wti.shape

In [ ]:
wti.head()

So we'll calculate timeliness and for each country-indicator pair we'll take the minimum.

In [ ]:
wti['timeliness'] = (2025 - wti['DataYear']).astype('Int64')

In [ ]:
# get min timeliness

wti_timeliness = wti.groupby(["ShortName", "Code", "RegionName", "Code description", "IsoCode", "wti_group", 'new_group', 'CountryType', 'WB_Income'], as_index=False)['timeliness'].min()
wti_timeliness.head()

In [ ]:
# with the transformation above we get one line per country-indicator pair -- here we can see it's also the most recent one, corresponding to 2023
wti_timeliness[(wti_timeliness.ShortName=='Afghanistan') & (wti_timeliness.Code =='i112')]

In [ ]:
wti_timeliness[wti_timeliness.ShortName=='Afghanistan']

In [ ]:
wti.timeliness.describe()

Here we transform the data so there is one timeliness value per country-indicator pair. This data contains NaNs

In [ ]:
pivoted_t = wti_timeliness.pivot(index=['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType', 'WB_Income'], columns='Code', values=['timeliness'])
# Replace NaNs with 5 because max value for available data is 2024-2020 =4
#pivoted_t = pivoted_t.fillna(5)

# Flatten and rename the columns
pivoted_t.columns = [
    f"{indicator}_{suffix}"
    for suffix, indicator in pivoted_t.columns
]

pivoted_t.reset_index(inplace=True)

pivoted_t.head()

In [ ]:
pivoted_t.ShortName.nunique()

In [ ]:
# add a single average indicator of timeliness so we can plot it against availability

pivoted_t['timeliness'] = pivoted_t[['i112_timeliness', 'i135tfb_timeliness',
       'i136mwi_timeliness', 'i271_timeliness', 'i271G_timeliness',
       'i271G5_pop_timeliness', 'i271GA_timeliness', 'i271mw_timeliness',
       'i4213_256to2_timeliness', 'i4213_2to10_timeliness',
       'i4213_G10_timeliness', 'i4213tfbb_timeliness', 'i4214l_timeliness',
       'i4214u_timeliness']].mean(axis=1)

pivoted_t['timeliness'].describe()

In [ ]:
# just out of curiosity checking which were these least timely countries
pivoted_t[pivoted_t.timeliness==5]

In [ ]:
pivoted_t.columns

In [ ]:
pivoted_t.i4213_256to2_timeliness.describe()

In [ ]:
pivoted_t[pivoted_t.ShortName == 'Chile']

In [ ]:
wti.wti_group.unique()

In [ ]:
wti[(wti.ShortName=='Chile')& (wti.wti_group=='D. Operators_wholesale') ]

In [ ]:
# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    #'#696969'
    #'#999999',   # Grey
    ]

rename_dict = {
 'i112_timeliness': 'A i112',
 'i135tfb_timeliness': 'C i135tfb',
 'i136mwi_timeliness': 'C i136mwi',
 'i271_timeliness': 'A i271',
 'i271G_timeliness': 'B i271G',
 'i271G5_pop_timeliness': 'B i271G5_pop',
 'i271GA_timeliness': 'B i271GA',
 'i271mw_timeliness': 'A i271mw',
 'i4213_256to2_timeliness': 'E i4213_256to2',
 'i4213_2to10_timeliness': 'E i4213_2to10',
 'i4213_G10_timeliness': 'E i4213_G10',
 'i4213tfbb_timeliness': 'A i4213tfbb',
 'i4214l_timeliness': 'D i4214l',
 'i4214u_timeliness': 'D i4214u'
}

pivoted_t[['RegionName','i112_timeliness',
       'i135tfb_timeliness', 'i136mwi_timeliness', 'i271_timeliness',
       'i271G_timeliness', 'i271G5_pop_timeliness', 'i271GA_timeliness',
       'i271mw_timeliness', 'i4213_256to2_timeliness',
       'i4213_2to10_timeliness', 'i4213_G10_timeliness',
       'i4213tfbb_timeliness', 'i4214l_timeliness', 'i4214u_timeliness']].rename(
     columns=rename_dict).groupby('RegionName').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Europe countries', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  color=custom_colors,
                                                  width=0.7)
#plt.title('Timeliness in 2024: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
# Define your custom order
custom_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']
#, 'Other Economies']

rename_dict = {
 'i112_timeliness': 'A i112',
 'i135tfb_timeliness': 'C i135tfb',
 'i136mwi_timeliness': 'C i136mwi',
 'i271_timeliness': 'A i271',
 'i271G_timeliness': 'B i271G',
 'i271G5_pop_timeliness': 'B i271G5_pop',
 'i271GA_timeliness': 'B i271GA',
 'i271mw_timeliness': 'A i271mw',
 'i4213_256to2_timeliness': 'E i4213_256to2',
 'i4213_2to10_timeliness': 'E i4213_2to10',
 'i4213_G10_timeliness': 'E i4213_G10',
 'i4213tfbb_timeliness': 'A i4213tfbb',
 'i4214l_timeliness': 'D i4214l',
 'i4214u_timeliness': 'D i4214u'
}

pivoted_t[['WB_Income','i112_timeliness',
       'i135tfb_timeliness', 'i136mwi_timeliness', 'i271_timeliness',
       'i271G_timeliness', 'i271G5_pop_timeliness', 'i271GA_timeliness',
       'i271mw_timeliness', 'i4213_256to2_timeliness',
       'i4213_2to10_timeliness', 'i4213_G10_timeliness',
       'i4213tfbb_timeliness', 'i4214l_timeliness', 'i4214u_timeliness']].rename(
     columns=rename_dict).groupby('WB_Income').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Upper middle income', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  #color=custom_colors,
                                                  width=0.7)
#plt.title('Timeliness in 2024: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
# Define your custom order
custom_order = ['Developed-OECD', 'Other developing and transition', 'SIDS + LDC']
#, 'Other Economies']

rename_dict = {
 'i112_timeliness': 'A i112',
 'i135tfb_timeliness': 'C i135tfb',
 'i136mwi_timeliness': 'C i136mwi',
 'i271_timeliness': 'A i271',
 'i271G_timeliness': 'B i271G',
 'i271G5_pop_timeliness': 'B i271G5_pop',
 'i271GA_timeliness': 'B i271GA',
 'i271mw_timeliness': 'A i271mw',
 'i4213_256to2_timeliness': 'E i4213_256to2',
 'i4213_2to10_timeliness': 'E i4213_2to10',
 'i4213_G10_timeliness': 'E i4213_G10',
 'i4213tfbb_timeliness': 'A i4213tfbb',
 'i4214l_timeliness': 'D i4214l',
 'i4214u_timeliness': 'D i4214u'
}

pivoted_t[['new_group','i112_timeliness',
       'i135tfb_timeliness', 'i136mwi_timeliness', 'i271_timeliness',
       'i271G_timeliness', 'i271G5_pop_timeliness', 'i271GA_timeliness',
       'i271mw_timeliness', 'i4213_256to2_timeliness',
       'i4213_2to10_timeliness', 'i4213_G10_timeliness',
       'i4213tfbb_timeliness', 'i4214l_timeliness', 'i4214u_timeliness']].rename(
     columns=rename_dict).groupby('new_group').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Developed-OECD', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  #color=custom_colors,
                                                  width=0.7)
#plt.title('Timeliness in 2024: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
# Define your custom order
custom_order = ['Developed', 'Developing']
#, 'Other Economies']

rename_dict = {
 'i112_timeliness': 'A i112',
 'i135tfb_timeliness': 'C i135tfb',
 'i136mwi_timeliness': 'C i136mwi',
 'i271_timeliness': 'A i271',
 'i271G_timeliness': 'B i271G',
 'i271G5_pop_timeliness': 'B i271G5_pop',
 'i271GA_timeliness': 'B i271GA',
 'i271mw_timeliness': 'A i271mw',
 'i4213_256to2_timeliness': 'A i4213_256to2',
 'i4213_2to10_timeliness': 'A i4213_2to10',
 'i4213_G10_timeliness': 'A i4213_G10',
 'i4213tfbb_timeliness': 'A i4213tfbb',
 'i4214l_timeliness': 'D i4214l',
 'i4214u_timeliness': 'D i4214u'
}

pivoted_t[['CountryType','i112_timeliness',
       'i135tfb_timeliness', 'i136mwi_timeliness', 'i271_timeliness',
       'i271G_timeliness', 'i271G5_pop_timeliness', 'i271GA_timeliness',
       'i271mw_timeliness', 'i4213_256to2_timeliness',
       'i4213_2to10_timeliness', 'i4213_G10_timeliness',
       'i4213tfbb_timeliness', 'i4214l_timeliness', 'i4214u_timeliness']].rename(
     columns=rename_dict).groupby('CountryType').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Developing', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  #color=custom_colors,
                                                  width=0.7)
plt.title('Timeliness in 2024: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

## Charts for country profile

In [ ]:
pivoted_t[pivoted_t.ShortName=='Madagascar'][['RegionName','i112_timeliness',
       'i135tfb_timeliness', 'i136mwi_timeliness', 'i271_timeliness',
       'i271G_timeliness', 'i271G5_pop_timeliness', 'i271GA_timeliness',
       'i271mw_timeliness', 'i4213_256to2_timeliness',
       'i4213_2to10_timeliness', 'i4213_G10_timeliness',
       'i4213tfbb_timeliness', 'i4214l_timeliness', 'i4214u_timeliness']]

In [ ]:
# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    #'#696969'
    #'#999999',   # Grey
    ]

rename_dict = {
 'i112_timeliness': 'A i112',
 'i135tfb_timeliness': 'C i135tfb',
 'i136mwi_timeliness': 'C i136mwi',
 'i271_timeliness': 'A i271',
 'i271G_timeliness': 'B i271G',
 'i271G5_pop_timeliness': 'B i271G5_pop',
 'i271GA_timeliness': 'B i271GA',
 'i271mw_timeliness': 'A i271mw',
 'i4213_256to2_timeliness': 'E i4213_256to2',
 'i4213_2to10_timeliness': 'E i4213_2to10',
 'i4213_G10_timeliness': 'E i4213_G10',
 'i4213tfbb_timeliness': 'A i4213tfbb',
 'i4214l_timeliness': 'D i4214l',
 'i4214u_timeliness': 'D i4214u'
}

pivoted_t[pivoted_t.ShortName=='Madagascar'][['RegionName','i112_timeliness',
       'i135tfb_timeliness', 'i136mwi_timeliness', 'i271_timeliness',
       'i271G_timeliness', 'i271G5_pop_timeliness', 'i271GA_timeliness',
       'i271mw_timeliness', 'i4213_256to2_timeliness',
       'i4213_2to10_timeliness', 'i4213_G10_timeliness',
       'i4213tfbb_timeliness', 'i4214l_timeliness', 'i4214u_timeliness']].rename(
     columns=rename_dict).groupby('RegionName').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Africa', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  color=custom_colors,
                                                  width=0.7)
#plt.title('Timeliness in 2024: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.5),
    ncol=7,
    fontsize=12
)
plt.legend().remove()

plt.tight_layout()
plt.show()

## Group-wise calculations for timeliness (by group of indicators)

Now group the data so we can report timeliness for the categories of indicators

In [ ]:
wti_timeliness.head()

In [ ]:
# Make calculations for the group of indicators - take the minimum per group

wti_timeliness.groupby(["ShortName", "RegionName", "wti_group", "IsoCode", 'new_group', 'CountryType', 'WB_Income'], as_index=False)["timeliness"].min().head()

In [ ]:
# check the range of timeliness
wti_timeliness.groupby(["ShortName", "RegionName", "wti_group", "IsoCode", 'new_group', 'CountryType', 'WB_Income'], as_index=False)["timeliness"].min().timeliness.describe()

In [ ]:
wti_time_group = wti_timeliness.groupby(["ShortName", "RegionName", "wti_group", "IsoCode", 'new_group', 'CountryType', 'WB_Income'], as_index=False)["timeliness"].min()

# the percentage column is only needed at the later stage when computing the sub-index


pivoted_t_group = wti_time_group.pivot(index=['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType', 'WB_Income'], columns='wti_group', values=['timeliness'])

# Pivot back my data so I can plot it
# Replace NaNs with 6 because the data must be older than 2025-2020 =5 years
# pivoted_t_group = pivoted_t_group.fillna(6)

# Flatten and rename the columns
pivoted_t_group.columns = [
    f"{indicator}_{suffix}"
    for suffix, indicator in pivoted_t_group.columns
]

pivoted_t_group.reset_index(inplace=True)

pivoted_t_group.head()

In [ ]:
pivoted_t_group.columns

In [ ]:
pivoted_t_group[['ShortName', 'RegionName','A. Contracting_records_timeliness',
       'B. Cell_tower_coverage_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness',
       'E. Contracting_fixed_broadband_speed_timeliness']][pivoted_t_group.RegionName== 'CIS countries']

In [ ]:
pivoted_t_group[['A. Contracting_records_timeliness',
       'B. Cell_tower_coverage_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness',
       'E. Contracting_fixed_broadband_speed_timeliness']].describe()

In [ ]:
# these are the countries with the lowest timeliness values (5) per group of indicator, not accounting for missing values
pivoted_t_group[pivoted_t_group[['A. Contracting_records_timeliness',
       'B. Cell_tower_coverage_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness',
       'E. Contracting_fixed_broadband_speed_timeliness']].ge(5).any(axis=1)]

In [ ]:
# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    #'#696969'
    #'#999999',   # Grey
    ]

rename_dict = {
'B. Cell_tower_coverage_timeliness': "B Cell tower coverage",
'A. Contracting_records_timeliness':"A Contracting records",
'C. Operator_traffic_timeliness':"C Operator traffic",
'D. Operators_wholesale_timeliness':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_timeliness': "E Contracting fixed-broadband speed"
}

pivoted_t_group[['RegionName','B. Cell_tower_coverage_timeliness',
       'A. Contracting_records_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness', 'E. Contracting_fixed_broadband_speed_timeliness']].rename(
     columns=rename_dict).groupby('RegionName').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Europe countries', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  color=custom_colors,
                                                  width=0.7)
#plt.title('Timeliness in 2025: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.6)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.1),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    ]

rename_dict = {
'B. Cell_tower_coverage_timeliness': "B Cell tower coverage",
'A. Contracting_records_timeliness':"A Contracting records",
'C. Operator_traffic_timeliness':"C Operator traffic",
'D. Operators_wholesale_timeliness':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_timeliness': "E Contracting fixed-broadband speed"
}

subset = pivoted_t_group[['RegionName','B. Cell_tower_coverage_timeliness',
       'A. Contracting_records_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness', 'E. Contracting_fixed_broadband_speed_timeliness']].rename(
     columns=rename_dict)

# Calculate means and counts
means = subset.groupby('RegionName').mean(numeric_only=True).T
counts = subset.groupby('RegionName').count().T

# Sort by Europe countries
means_sorted = means.sort_values(by='Europe countries', ascending=True)[custom_order]
counts_sorted = counts.loc[means_sorted.index][custom_order]

# Set width factor range
min_width_factor = 0.3
max_width_factor = 1.0

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

# Plot each region separately to control width
x = np.arange(len(means_sorted.index))
base_width = 0.6 / len(custom_order)

for i, region in enumerate(custom_order):
    # Calculate width for each bar WITHIN THIS GROUP
    group_max_count = counts_sorted[region].max()

    # Normalize within the group
    width_factors = counts_sorted[region] / group_max_count * (max_width_factor - min_width_factor) + min_width_factor

    # Plot bars with individual widths
    for j, (indicator, value) in enumerate(means_sorted[region].items()):
        bar_width = base_width * width_factors.iloc[j]
        bar_x = x[j] + i * base_width - (len(custom_order) - 1) * base_width / 2

        ax.bar(
            bar_x,
            value,
            bar_width,
            color=custom_colors[i]
        )

        # Add text label on top of bar
        count = int(counts_sorted[region].iloc[j])
        if count > 0:
            ax.text(
                bar_x,
                value,
                f'n={count}',
                ha='center',
                va='bottom',
                fontsize=8,
                rotation=0
            )

ax.set_xlabel('', fontsize=14)
ax.set_ylabel('number of years', fontsize=14)
#plt.title('Timeliness in 2025: time lag of latest available data point in number of years (available data only)')
ax.set_xticks(x)
ax.set_xticklabels(means_sorted.index, rotation=45, fontsize=14, ha='right')
ax.tick_params(axis='y', labelsize=14)

# Create custom legend
legend_handles = [Patch(facecolor=custom_colors[i], label=region)
                  for i, region in enumerate(custom_order)]

plt.subplots_adjust(bottom=0.7)
plt.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.6),
    ncol=1,
    fontsize=14
)

plt.tight_layout()
plt.show()



In [ ]:
# Define your custom order
custom_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']
#, 'Other Economies']

rename_dict = {
'B. Cell_tower_coverage_timeliness': "B Cell tower coverage",
'A. Contracting_records_timeliness':"A Contracting records",
'C. Operator_traffic_timeliness':"C Operator traffic",
'D. Operators_wholesale_timeliness':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_timeliness': "E Contracting fixed-broadband speed"
}

pivoted_t_group[['WB_Income','B. Cell_tower_coverage_timeliness',
       'A. Contracting_records_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness', 'E. Contracting_fixed_broadband_speed_timeliness']].rename(
     columns=rename_dict).groupby('WB_Income').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Upper middle income', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  #color=custom_colors,
                                                  width=0.7)
#plt.title('Timeliness in 2025: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.6)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.1),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Define your custom order
custom_order = ['High income', 'Upper middle income', 'Lower middle income', 'Low income']
#, 'Other Economies']

rename_dict = {
'B. Cell_tower_coverage_timeliness': "B Cell tower coverage",
'A. Contracting_records_timeliness':"A Contracting records",
'C. Operator_traffic_timeliness':"C Operator traffic",
'D. Operators_wholesale_timeliness':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_timeliness': "E Contracting fixed-broadband speed"
}

custom_colors = ['#1f77b4',  # Blue
'#ff7f0e' , # Orange
'#2ca02c',  # Green
'#d62728'  # Red
]

subset = pivoted_t_group[['WB_Income','B. Cell_tower_coverage_timeliness',
       'A. Contracting_records_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness', 'E. Contracting_fixed_broadband_speed_timeliness']].rename(
     columns=rename_dict)

# Calculate means and counts
means = subset.groupby('WB_Income').mean(numeric_only=True).T
counts = subset.groupby('WB_Income').count().T

# Sort by Europe countries
means_sorted = means.sort_values(by='Upper middle income', ascending=True)[custom_order]
counts_sorted = counts.loc[means_sorted.index][custom_order]

# Set width factor range
min_width_factor = 0.3
max_width_factor = 1.0

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

# Plot each region separately to control width
x = np.arange(len(means_sorted.index))
base_width = 0.6 / len(custom_order)

for i, region in enumerate(custom_order):
    # Calculate width for each bar WITHIN THIS GROUP
    group_max_count = counts_sorted[region].max()

    # Normalize within the group
    width_factors = counts_sorted[region] / group_max_count * (max_width_factor - min_width_factor) + min_width_factor

    # Plot bars with individual widths
    for j, (indicator, value) in enumerate(means_sorted[region].items()):
        bar_width = base_width * width_factors.iloc[j]
        bar_x = x[j] + i * base_width - (len(custom_order) - 1) * base_width / 2

        ax.bar(
            bar_x,
            value,
            bar_width,
            color=custom_colors[i]
        )

        # Add text label on top of bar
        count = int(counts_sorted[region].iloc[j])
        if count > 0:
            ax.text(
                bar_x,
                value,
                f'n={count}',
                ha='center',
                va='bottom',
                fontsize=8,
                rotation=0
            )

ax.set_xlabel('', fontsize=14)
ax.set_ylabel('number of years', fontsize=14)
#plt.title('Timeliness in 2025: time lag of latest available data point in number of years (available data only)')
ax.set_xticks(x)
ax.set_xticklabels(means_sorted.index, rotation=45, fontsize=14, ha='right')
ax.tick_params(axis='y', labelsize=14)

# Create custom legend
legend_handles = [Patch(facecolor=custom_colors[i], label=region)
                  for i, region in enumerate(custom_order)] #

plt.subplots_adjust(bottom=0.7)
plt.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.6),
    ncol=1,
    fontsize=14
)

plt.tight_layout()
plt.show()



In [ ]:
# Define your custom order
custom_order = ['Developed-OECD', 'Other developing and transition', 'SIDS + LDC']
#, 'Other Economies']

rename_dict = {
'B. Cell_tower_coverage_timeliness': "B Cell tower coverage",
'A. Contracting_records_timeliness':"A Contracting records",
'C. Operator_traffic_timeliness':"C Operator traffic",
'D. Operators_wholesale_timeliness':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_timeliness': "E Contracting fixed-broadband speed"
}

pivoted_t_group[['new_group','B. Cell_tower_coverage_timeliness',
       'A. Contracting_records_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness', 'E. Contracting_fixed_broadband_speed_timeliness']].rename(
     columns=rename_dict).groupby('new_group').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Developed-OECD', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  #color=custom_colors,
                                                  width=0.7)
#plt.title('Timeliness in 2025: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.6)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.1),
    ncol=7,
    fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Define your custom order
custom_order = ['Developed-OECD', 'Other developing and transition', 'SIDS + LDC']
#, 'Other Economies']

rename_dict = {
'B. Cell_tower_coverage_timeliness': "B Cell tower coverage",
'A. Contracting_records_timeliness':"A Contracting records",
'C. Operator_traffic_timeliness':"C Operator traffic",
'D. Operators_wholesale_timeliness':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_timeliness': "E Contracting fixed-broadband speed"
}

custom_colors = ['#1f77b4',  # Blue
'#ff7f0e' , # Orange
'#2ca02c',  # Green
'#d62728'  # Red
]

subset = pivoted_t_group[['new_group','B. Cell_tower_coverage_timeliness',
       'A. Contracting_records_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness', 'E. Contracting_fixed_broadband_speed_timeliness']].rename(
     columns=rename_dict)

# Calculate means and counts
means = subset.groupby('new_group').mean(numeric_only=True).T
counts = subset.groupby('new_group').count().T

# Sort by Europe countries
means_sorted = means.sort_values(by='Other developing and transition', ascending=True)[custom_order]
counts_sorted = counts.loc[means_sorted.index][custom_order]

# Set width factor range
min_width_factor = 0.3
max_width_factor = 1.0

# Create figure
fig, ax = plt.subplots(figsize=(14, 6))

# Plot each region separately to control width
x = np.arange(len(means_sorted.index))
base_width = 0.6 / len(custom_order)

for i, region in enumerate(custom_order):
    # Calculate width for each bar WITHIN THIS GROUP
    group_max_count = counts_sorted[region].max()

    # Normalize within the group
    width_factors = counts_sorted[region] / group_max_count * (max_width_factor - min_width_factor) + min_width_factor

    # Plot bars with individual widths
    for j, (indicator, value) in enumerate(means_sorted[region].items()):
        bar_width = base_width * width_factors.iloc[j]
        bar_x = x[j] + i * base_width - (len(custom_order) - 1) * base_width / 2

        ax.bar(
            bar_x,
            value,
            bar_width,
            color=custom_colors[i]
        )

        # Add text label on top of bar
        count = int(counts_sorted[region].iloc[j])
        if count > 0:
            ax.text(
                bar_x,
                value,
                f'n={count}',
                ha='center',
                va='bottom',
                fontsize=8,
                rotation=0
            )

ax.set_xlabel('', fontsize=14)
ax.set_ylabel('number of years', fontsize=14)
#plt.title('Timeliness in 2025: time lag of latest available data point in number of years (available data only)')
ax.set_xticks(x)
ax.set_xticklabels(means_sorted.index, rotation=45, fontsize=14, ha='right')
ax.tick_params(axis='y', labelsize=14)

# Create custom legend
legend_handles = [Patch(facecolor=custom_colors[i], label=region)
                  for i, region in enumerate(custom_order)] #

plt.subplots_adjust(bottom=0.7)
plt.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.0),
    ncol=1,
    fontsize=14
)

plt.tight_layout()
plt.show()



In [ ]:
# for oecd countries, the only data available is the one below and timeliness averages to 0⁄
wti_timeliness[(wti_timeliness.new_group=='Developed-OECD') & (wti_timeliness.wti_group=='D. Operators_wholesale')]

In [ ]:
pivoted_t[pivoted_t.ShortName=='Afghanistan']

In [ ]:
wti_timeliness[wti_timeliness.ShortName=='Afghanistan']

In [ ]:
wti_counts[wti_counts.ShortName=='Afghanistan']

In [ ]:
wti[wti.ShortName=='Afghanistan']

## Charts for country profile

In [ ]:
pivoted_t_group[pivoted_t_group.ShortName=='Madagascar'][['RegionName','B. Cell_tower_coverage_timeliness',
       'A. Contracting_records_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness', 'E. Contracting_fixed_broadband_speed_timeliness']]

In [ ]:
# Define your custom order
custom_order = ['Europe countries','CIS countries', 'Asia & Pacific', 'The Americas', 'Arab States', 'Africa']
#, 'Other Economies']

custom_colors = [
    '#0072B2',  # Blue
    '#CC79A7',  # Pink/Purple
    '#009E73',  # Green
    '#F0E442',  # Yellow
    '#56B4E9',  # Light Blue
    '#E69F00'  # Orange
    #'#696969'
    #'#999999',   # Grey
    ]

rename_dict = {
'B. Cell_tower_coverage_timeliness': "B Cell tower coverage",
'A. Contracting_records_timeliness':"A Contracting records",
'C. Operator_traffic_timeliness':"C Operator traffic",
'D. Operators_wholesale_timeliness':"D Operators wholesale",
'E. Contracting_fixed_broadband_speed_timeliness': "E Contracting fixed-broadband speed"
}

pivoted_t_group[pivoted_t_group.ShortName=='Madagascar'][['RegionName','B. Cell_tower_coverage_timeliness',
       'A. Contracting_records_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness', 'E. Contracting_fixed_broadband_speed_timeliness']].rename(
     columns=rename_dict).groupby('RegionName').mean(numeric_only=True).reindex(custom_order).T.sort_values(by='Africa', ascending=True).plot(kind='bar',
                                                  figsize=(15, 6),
                                                  color=custom_colors,
                                                  width=0.7)
#plt.title('Timeliness in 2025: time lag of latest available data point in number of years (available data only)')
plt.xlabel('')
plt.ylabel('number of years', fontsize=14)
plt.xticks(rotation=45, fontsize=14)
plt.yticks(fontsize=14)
#plt.legend(title='', bbox_to_anchor=(1.05, 1), loc='upper left')

# adjust bottom margin to make space for legend
plt.subplots_adjust(bottom=0.6)  # increase bottom space

# Step 1: Get current handles and labels
handles, labels = plt.gca().get_legend_handles_labels()

# Step 2: Create a dict for easy lookup
label_to_handle = dict(zip(labels, handles))

# Step 3: Reconstruct the legend using custom order
plt.legend(
    [label_to_handle[label] for label in custom_order],
    custom_order,
    loc='upper center',
    bbox_to_anchor=(0.5, -1.1),
    ncol=7,
    fontsize=12
)
plt.legend().remove()

plt.tight_layout()
plt.show()

# WTI indicators results by country

In [ ]:
pivoted_group.columns

In [ ]:
pivoted_t_group.columns

In [ ]:
pivoted_group[['ShortName', 'IsoCode', 'RegionName', 'new_group',
       'WB_Income', 'A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
       'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
       'E. Contracting_fixed_broadband_speed_5Y']].merge(pivoted_t_group[['IsoCode', 'A. Contracting_records_timeliness',
       'B. Cell_tower_coverage_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness',
       'E. Contracting_fixed_broadband_speed_timeliness']], on='IsoCode', how='left')

In [ ]:
results_wti_capacity = pivoted_group[['ShortName', 'IsoCode', 'RegionName', 'new_group',
       'WB_Income', 'A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
       'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
       'E. Contracting_fixed_broadband_speed_5Y']].merge(pivoted_t_group[['IsoCode', 'A. Contracting_records_timeliness',
       'B. Cell_tower_coverage_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness',
       'E. Contracting_fixed_broadband_speed_timeliness']], on='IsoCode', how='left')

In [ ]:
results_wti_capacity.to_excel('wti_capacity_country.xlsx', index=False)

## Comparing availability to timeliness

In [ ]:
merged = pd.merge(pivoted[['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType',
       'WB_Income','availability_5Y']], pivoted_t[['IsoCode', 'timeliness']], on='IsoCode', how='left')

merged.ShortName.nunique()

In [ ]:
import matplotlib.patches as mpatches

# Start plotting
plt.figure(figsize=(15, 8))

# Assign a color to each income group
group_colors = {
    #'n.a.': '#999999',
    'Low income': '#d73027',
    'Lower middle income': '#fdd835',
    'Upper middle income': '#1a9850',
    'High income': '#1e88e5'
}
merged['group_color'] = merged['WB_Income'].map(group_colors)

# Define your columns
x_column = 'availability_5Y'
y_column = 'timeliness'
label_column='IsoCode'

# Scatter plot with manually assigned colors
plt.scatter(
    merged[x_column], merged[y_column],
    c=merged['group_color'],
    s=70, edgecolors='k', linewidths=0.1, alpha=0.7
)

# Add point labels
for i, label in enumerate(merged[label_column]):
    plt.text(merged[x_column].iloc[i]+0.01, merged[y_column].iloc[i]+0.01, label, fontsize=8)


# Add axis labels and title
plt.xlabel(x_column.upper())
plt.ylabel(y_column.upper())


# Add legend
patches = [mpatches.Patch(color=color, label=label) for label, color in group_colors.items()]
plt.legend(handles=patches, title='World Bank income group', frameon=False)

plt.show()

Sanity check 1: all countries that have 100% availability, have timeliness

In [ ]:
merged [merged.availability_5Y==5]

In [ ]:
# list of developing, transition and SIDS+LDC who do well

merged[(merged.timeliness<1.5) & (merged.availability_5Y>=4.9) & (merged['new_group'].isin(['Other developing and transition', 'SIDS + LDC']))].sort_values(by='ShortName', ascending=True)

In [ ]:
# countries that could be considered of priority for capacity building
merged [(merged.timeliness>=2) & (merged.availability_5Y<4)].sort_values(by='IsoCode', ascending=True)

In [ ]:
!pip install pycountry
import pycountry

# Create a dictionary mapping ISO3 to ISO2 codes
iso3_to_iso2 = {country.alpha_3: country.alpha_2 for country in pycountry.countries}

# Map the ISO3 codes to ISO2 codes
merged['iso2'] = merged['IsoCode'].map(iso3_to_iso2)

In [ ]:
import matplotlib.patches as mpatches

# Start plotting
plt.figure(figsize=(15, 8))

# Assign a color to each income group
group_colors = {
    #'n.a.': '#999999',
    'Low income': '#d73027',
    'Lower middle income': '#fdd835',
    'Upper middle income': '#1a9850',
    'High income': '#1e88e5'
}
merged['group_color'] = merged['WB_Income'].map(group_colors)

# Define your columns
x_column = 'availability_5Y'
y_column = 'timeliness'
label_column='iso2'

# Scatter plot with manually assigned colors
plt.scatter(
    merged[x_column], merged[y_column],
    c=merged['group_color'],
    s=70, edgecolors='k', linewidths=0.1, alpha=0.7
)

# Add point labels
#for i, label in enumerate(merged[label_column]):
    #plt.text(merged[x_column].iloc[i]+0.01, merged[y_column].iloc[i]+0.01, label, fontsize=8)

# Create text objects
texts = []
for i, row in merged.iterrows():
    texts.append(
        plt.text(row['availability_5Y'],
                 row['timeliness'],
                 row['IsoCode'],
                 fontsize=8)
    )

adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))


# Add axis labels and title
plt.xlabel(x_column.upper())
plt.ylabel(y_column.upper())


# Add legend
patches = [mpatches.Patch(color=color, label=label) for label, color in group_colors.items()]
plt.legend(handles=patches, title='World Bank income group', frameon=False)

plt.show()

Do this again, but for those with availability below 4

In [ ]:
merged[merged.availability_5Y<4].sort_values(by='IsoCode', ascending=True)

In [ ]:
import matplotlib.patches as mpatches

# Start plotting
plt.figure(figsize=(15, 8))

# Assign a color to each income group
group_colors = {
    #'n.a.': '#999999',
    'Low income': '#d73027',
    'Lower middle income': '#fdd835',
    'Upper middle income': '#1a9850',
    'High income': '#1e88e5'
}
merged['group_color'] = merged['WB_Income'].map(group_colors)

# Define your columns
x_column = 'availability_5Y'
y_column = 'timeliness'
label_column='iso2'

# Scatter plot with manually assigned colors
plt.scatter(
    merged[merged.availability_5Y<4][x_column], merged[merged.availability_5Y<4][y_column],
    c=merged[merged.availability_5Y<4]['group_color'],
    s=70, edgecolors='k', linewidths=0.1, alpha=0.7
)

# Add point labels
#for i, label in enumerate(merged[label_column]):
    #plt.text(merged[x_column].iloc[i]+0.01, merged[y_column].iloc[i]+0.01, label, fontsize=8)

# Create text objects
texts = []
for i, row in merged[merged.availability_5Y<4].iterrows():
    texts.append(
        plt.text(row['availability_5Y'],
                 row['timeliness'],
                 row['IsoCode'],
                 fontsize=10)
    )

adjust_text(texts, arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))


# Add axis labels and title
plt.xlabel('Availability: number of data points available between 2020-2024')
plt.ylabel('Timeliness: age of latest data')


# Add legend
patches = [mpatches.Patch(color=color, label=label) for label, color in group_colors.items()]
plt.legend(handles=patches, title='World Bank income group', frameon=False)

plt.show()

## Group-wise find priority countries for assistance

By indicator group, find countries that have unsatisfactory timeliness and low availability by group of indicators

In [ ]:
merged_group = pd.merge(pivoted_group[['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType',
       'WB_Income', 'A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
       'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
       'E. Contracting_fixed_broadband_speed_5Y',]], pivoted_t_group[['IsoCode', 'A. Contracting_records_timeliness',
       'B. Cell_tower_coverage_timeliness', 'C. Operator_traffic_timeliness',
       'D. Operators_wholesale_timeliness',
       'E. Contracting_fixed_broadband_speed_timeliness']],
        on='IsoCode', how='left')

merged_group.ShortName.nunique()

In [ ]:
merged_group[['A. Contracting_records_timeliness', 'B. Cell_tower_coverage_5Y', 'C. Operator_traffic_5Y','D. Operators_wholesale_5Y','E. Contracting_fixed_broadband_speed_5Y', ]].describe()

In [ ]:
# now checking for cell tower coverage indicators

merged_group[['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType',
       'WB_Income', 'B. Cell_tower_coverage_5Y', 'B. Cell_tower_coverage_timeliness']][(merged_group['B. Cell_tower_coverage_5Y']<3) & (merged_group['B. Cell_tower_coverage_timeliness']>1)].sort_values(by='ShortName', ascending=True)

In [ ]:
# now checking for Contracting_fixed_broadband_speed


merged_group[['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType',
       'WB_Income', 'E. Contracting_fixed_broadband_speed_5Y', 'E. Contracting_fixed_broadband_speed_timeliness']][(merged_group['E. Contracting_fixed_broadband_speed_5Y']<3) & (merged_group['E. Contracting_fixed_broadband_speed_timeliness']>1)].sort_values(by='ShortName', ascending=True)

In [ ]:
# now checking for Contracting_records



merged_group[['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType',
       'WB_Income', 'A. Contracting_records_5Y', 'A. Contracting_records_timeliness']][(merged_group['A. Contracting_records_5Y']<3) & (merged_group['A. Contracting_records_timeliness']>1)].sort_values(by='ShortName', ascending=True)

In [ ]:
# now checking for Operator_traffic



merged_group[['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType',
       'WB_Income', 'C. Operator_traffic_5Y', 'C. Operator_traffic_timeliness']][(merged_group['C. Operator_traffic_5Y']<3) & (merged_group['C. Operator_traffic_timeliness']>1)].sort_values(by='ShortName', ascending=True)

In [ ]:
# now checking for Operators_wholesale


merged_group[['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType',
       'WB_Income', 'D. Operators_wholesale_5Y', 'D. Operators_wholesale_timeliness']][(merged_group['D. Operators_wholesale_5Y']<3) & (merged_group['D. Operators_wholesale_timeliness']>1)].sort_values(by='ShortName', ascending=True)

# REDO the analysis but with missing data assumptions, so that we can see the countries with no data

In [ ]:
pivoted = wti_counts.pivot(index=['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType', 'WB_Income'], columns='Code', values=['5Y'])
# Replace NaNs with 0
pivoted = pivoted.fillna(0)

# Flatten and rename the columns
pivoted.columns = [
    f"{indicator}_{suffix}"
    for suffix, indicator in pivoted.columns
]

pivoted.reset_index(inplace=True)

pivoted.head()

In [ ]:
## this dataset contains imputed 0s as in 0 data points available
# we get countries with 0 data points in the last 5 years for every indicator
pivoted.describe()


In [ ]:
wti_counts_group = wti_counts.groupby(["ShortName", "RegionName", "wti_group", "IsoCode", 'new_group', 'CountryType', 'WB_Income'], as_index=False)["5Y"].max()

# add percentage column

wti_counts_group['5Y_p'] = wti_counts_group['5Y'] / 5 *100

pivoted_group = wti_counts_group.pivot(index=['ShortName', 'RegionName', 'IsoCode', 'new_group', 'CountryType', 'WB_Income'], columns='wti_group', values=['5Y'])

# Replace NaNs with 0
pivoted_group = pivoted_group.fillna(0)

# Flatten and rename the columns
pivoted_group.columns = [
    f"{indicator}_{suffix}"
    for suffix, indicator in pivoted_group.columns
]

pivoted_group.reset_index(inplace=True)

pivoted_group.head()


In [ ]:
# this shows countries with 0 records over the past 5 years by indicator group
pivoted_group.describe()

In [ ]:
pivoted_group[['ShortName', 'RegionName', 'IsoCode', 'new_group', 'WB_Income', 'A. Contracting_records_5Y']][pivoted_group['A. Contracting_records_5Y']<=1].sort_values(by='RegionName').reset_index(drop=True)

In [ ]:
pivoted_group[
    ['ShortName', 'RegionName', 'IsoCode', 'new_group', 'WB_Income',
     'B. Cell_tower_coverage_5Y']
][
    pivoted_group['B. Cell_tower_coverage_5Y'] <= 1
].sort_values(
    by=['RegionName', 'ShortName']
).reset_index(
    drop=True
)

In [ ]:
pivoted_group[
    ['ShortName', 'RegionName', 'IsoCode', 'new_group', 'WB_Income',
     'C. Operator_traffic_5Y']
][
    pivoted_group['C. Operator_traffic_5Y'] < 1
].sort_values(
    by=['RegionName', 'ShortName']
).reset_index(
    drop=True
)

In [ ]:
pivoted_group[
    ['ShortName', 'RegionName', 'IsoCode', 'new_group', 'WB_Income',
     'D. Operators_wholesale_5Y']
][
    (pivoted_group['D. Operators_wholesale_5Y'] < 1) &
    (pivoted_group['new_group'] != 'Developed-OECD')
].sort_values(
    by=['RegionName', 'ShortName']
).reset_index(
    drop=True
)

In [ ]:
pivoted_group[
    ['ShortName', 'RegionName', 'IsoCode', 'new_group', 'WB_Income',
     'E. Contracting_fixed_broadband_speed_5Y']
][
    (pivoted_group['E. Contracting_fixed_broadband_speed_5Y'] < 1) &
    (pivoted_group['new_group'] != 'Developed-OECD')
].sort_values(
    by=['RegionName', 'ShortName']
).reset_index(
    drop=True,
)

now do for all the countries with plenty of conditions

In [ ]:
pivoted_group.columns

In [ ]:
pivoted_group[
    ['ShortName', 'RegionName', 'IsoCode', 'new_group', 'WB_Income',
     'A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
     'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
     'E. Contracting_fixed_broadband_speed_5Y']
][
    (pivoted_group['new_group'] != 'Developed-OECD') &
    (
        (pivoted_group['A. Contracting_records_5Y'] < 1) |
        (pivoted_group['B. Cell_tower_coverage_5Y'] < 1) |
        (pivoted_group['C. Operator_traffic_5Y'] < 1) |
        (pivoted_group['D. Operators_wholesale_5Y'] < 1) |
        (pivoted_group['E. Contracting_fixed_broadband_speed_5Y'] < 1)
    )
].sort_values(
    by=['RegionName', 'ShortName']
).reset_index(
    drop=True
)

# Hierarchical clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

In [ ]:
# this is my dataset
pivoted_group[
    ['ShortName', 'RegionName', 'IsoCode', 'new_group', 'WB_Income',
     'A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
     'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
     'E. Contracting_fixed_broadband_speed_5Y']
]

# extracting the features

X= pivoted_group[['A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
     'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
     'E. Contracting_fixed_broadband_speed_5Y']].values

country_names = pivoted_group['ShortName'].values

# Step 2: Perform hierarchical clustering
# 'ward' minimizes variance within clusters (usually works well)
# Other options: 'complete', 'average', 'single'
linkage_matrix = linkage(X, method='ward')

In [ ]:
# Step 3: Create a dendrogram to visualize
plt.figure(figsize=(15, 8))
dendrogram(
    linkage_matrix,
    labels=country_names,
    leaf_rotation=90,
    leaf_font_size=8
)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Country')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Cut the dendrogram to create clusters
# Choose number of clusters (try different values)
n_clusters = 4  # Adjust this based on what you see in the dendrogram

clusters = fcluster(linkage_matrix, n_clusters, criterion='maxclust')

# Add cluster labels to your dataframe
pivoted_group['Cluster'] = clusters

# Step 5: Analyze the clusters
print(f"\nNumber of countries per cluster:")
print(pivoted_group['Cluster'].value_counts().sort_index())

print(f"\nCluster characteristics (mean values):")
cluster_summary = pivoted_group.groupby('Cluster')[['A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
     'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
     'E. Contracting_fixed_broadband_speed_5Y']].mean()
print(cluster_summary.round(2))

In [ ]:
# Step 6: See which countries are in each cluster
for i in range(1, n_clusters + 1):
    print(f"\n--- Cluster {i} ---")
    countries_in_cluster = pivoted_group[pivoted_group['Cluster'] == i]['ShortName'].tolist()
    print(f"Countries ({len(countries_in_cluster)}): {', '.join(countries_in_cluster)}")


In [ ]:
# Step 7: Create a heatmap to visualize cluster patterns
plt.figure(figsize=(12, 8))
cluster_means = pivoted_group.groupby('Cluster')[['A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y',
     'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y',
     'E. Contracting_fixed_broadband_speed_5Y']].mean()
sns.heatmap(
    cluster_means.T,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    cbar_kws={'label': 'Average Score'}
)
plt.title('Average Data Availability by Cluster')
plt.xlabel('Cluster')
plt.ylabel('Indicator')
plt.tight_layout()
plt.show()

In [ ]:

# Calculate correlation matrix
correlation_matrix = pivoted_group[[ 'A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y', 'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']].corr()

# Create the correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix,
            annot=True,  # Show correlation values
            fmt='.2f',   # 2 decimal places
            cmap='coolwarm',  # Blue (negative) to Red (positive)
            center=0,    # Center the colormap at 0
            square=True,  # Make cells square
            linewidths=1,
            linecolor='white',
            cbar_kws={'label': 'Correlation Coefficient'},
            vmin=-1,     # Correlations range from -1 to 1
            vmax=1)

plt.title('Correlation Matrix: ICT Indicators', fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Country profile charts

In [ ]:
# prepping my data to draw a barcode plot

country_name = 'Madagascar'

years = [2020, 2021, 2022, 2023, 2024]

codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA', 'i136mwi',  'i135tfb','i4214l',
       'i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

new_names = ['A.i112 Fixed-telephone subscriptions', 'A.i271 Mobile-cellular telephone subscriptions',
             'A.i271mw Active mobile-broadband subscriptions', 'A.i4213tfbb Fixed-broadband subscriptions',
             'B.i271G % population covered by at least a 3G mobile network' , 'B.i271G5_pop % population covered by at least a 5G mobile network',
             'B.i271GA % population covered by at least an LTE/WiMAX mobile network',
             'C.i136mwi Mobile-broadband Internet traffic',  'C.i135tfb Fixed-broadband Internet traffic','D.i4214l Lit/equipped international bandwidth capacity',
             'D.i4214u International bandwidth usage', 'E.i4213_256to2 Fixed broadband, 256 kbit/s to less than 2 Mbit/s',
             'E.i4213_2to10 Fixed broadband, 2 to less than 10 Mbit/s', 'E.i4213_G10 Fixed broadband, equal to or above 10 Mbit/s ']

wti_country = wti[(wti.ShortName == country_name)].pivot(index='Code', columns='DataYear', values = 'ShortName').reindex(columns=years, index=codes).notna().astype(int).reset_index()

wti_country.Code=new_names

# Convert DataFrame to numpy array
code_array = wti_country.set_index( 'Code').to_numpy()

fig, ax = plt.subplots(figsize=(code_array.shape[1] * 1, code_array.shape[0] * 0.4))

# Show barcode
ax.imshow(code_array, cmap='binary', aspect='auto', interpolation='nearest')

ax.set_title(f'{country_name} Supply Side Data Availability by Year and Indicator', fontsize=14, pad=30)

# Set row labels (y-axis)
ax.set_yticks(np.arange(code_array.shape[0]))
ax.set_yticklabels(wti_country.set_index( 'Code').index)

# Set column labels (x-axis)
ax.set_xticks(np.arange(code_array.shape[1]))
ax.set_xticklabels(wti_country.set_index( 'Code').columns, rotation=0)
ax.xaxis.tick_top() # put year lables on top

# Optional: remove grid / frame
ax.set_xticks(np.arange(-.5, code_array.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-.5, code_array.shape[0], 1), minor=True)
ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
ax.tick_params(which='minor', bottom=False, left=False)

plt.show()


In [ ]:
year_columns = [2020, 2021, 2022, 2023, 2024]
wti_country[year_columns].sum().sum()/(14*5)*100

In [ ]:
year_columns = [2020, 2021, 2022, 2023, 2024]

# Calculate your percentage
completeness = wti_country[year_columns].sum().sum()/(14*5)*100

# Create gauge chart
fig, ax = plt.subplots(figsize=(8, 4), subplot_kw={'projection': 'polar'})

# Set up the gauge (semicircle)
theta = np.linspace(0, np.pi, 100)

# Background arc (gray)
ax.plot(theta, [1]*100, color='lightgray', linewidth=20)

# Colored arc based on percentage
# Define color zones
if completeness >= 75:
    color = '#2ecc71'  # Green
elif completeness >= 50:
    color = '#f39c12'  # Orange
else:
    color = '#e74c3c'  # Red

# Calculate how much of the arc to fill
theta_fill = np.linspace(0, np.pi * (completeness/100), 100)
ax.plot(theta_fill, [1]*100, color=color, linewidth=20)

# Add percentage text in center
ax.text(np.pi/2, 0.5, f'{completeness:.1f}%',
        ha='center', va='center', fontsize=20, fontweight='bold')
# Add calculation text in center
ax.text(np.pi/2, 0.3, f'{completeness*0.70:.0f}/70',
        ha='center', va='center', fontsize=20, fontweight='bold')
ax.text(np.pi/2, 0.1, f'{country_name} Data Points',
        ha='center', va='center', fontsize=12, color='gray')

# Customize appearance
ax.set_ylim(0, 1)
ax.set_theta_offset(np.pi)
ax.set_theta_direction(-1)
ax.set_xticks([])
ax.set_yticks([])
ax.spines['polar'].set_visible(False)
ax.grid(False)
plt.subplots_adjust(bottom=0.1, top=0.9)

plt.tight_layout()
plt.show()

In [ ]:


# Calculate sum for each year (excluding the 'Indicator' column)
year_columns = [2020, 2021, 2022, 2023, 2024]
year_sums = wti_country[year_columns].sum()

# Create line chart
plt.figure(figsize=(8.2, 3))
plt.plot(year_sums.index, year_sums.values, marker='o', linewidth=2, markersize=8)

plt.title(f'{country_name} Number of Available Indicators by Year', fontsize=14, pad=20)
# Set x-ticks to integers only
plt.xticks(year_columns)  # This ensures only your year values appear as ticks
#plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Indicators', fontsize=12)
#plt.grid(True, alpha=0.3)
plt.ylim(0, max(year_sums.values) + 1)  # Add some space at the top
plt.ylim(-1, max(year_sums.values) + 1)  # Starts at -1 instead of 0

plt.tight_layout()
plt.show()

In [ ]:
# Select the row for that country
country_name = 'Madagascar'
country_data = pivoted_group[pivoted_group['ShortName'] == country_name].drop('ShortName', axis=1)

plt.figure(figsize=(12, 3))
sns.heatmap(country_data[['A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y', 'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']],
            annot=True,
            fmt='.0f',
            cmap='RdYlGn',  # Red-Yellow-Green
            cbar_kws={'label': 'Average Availability'},
            linewidths=1,
            linecolor='white',
            vmin=0,
            vmax=5)

plt.title(f'{country_name} Availability by Indicator Group', fontsize=14, pad=20)
#plt.xlabel('Indicator', fontsize=12)
#plt.ylabel('')
plt.yticks([])
plt.xticks(rotation=45, ha='right')
#plt.tight_layout()
plt.show()

And ALL IN ONE

In [ ]:
# List of countries
countries = ['Madagascar', 'Sierra Leone', 'Mali']

In [ ]:
# Loop through each country
for country_name in countries:

    # prepping my data to draw a barcode plot


    years = [2020, 2021, 2022, 2023, 2024]

    codes = ['i112', 'i271', 'i271mw', 'i4213tfbb', 'i271G' , 'i271G5_pop', 'i271GA', 'i136mwi',  'i135tfb','i4214l',
       'i4214u', 'i4213_256to2', 'i4213_2to10', 'i4213_G10']

    new_names = ['A.i112 Fixed-telephone subscriptions', 'A.i271 Mobile-cellular telephone subscriptions',
             'A.i271mw Active mobile-broadband subscriptions', 'A.i4213tfbb Fixed-broadband subscriptions',
             'B.i271G % population covered by at least a 3G mobile network' , 'B.i271G5_pop % population covered by at least a 5G mobile network',
             'B.i271GA % population covered by at least an LTE/WiMAX mobile network',
             'C.i136mwi Mobile-broadband Internet traffic',  'C.i135tfb Fixed-broadband Internet traffic','D.i4214l Lit/equipped international bandwidth capacity',
             'D.i4214u International bandwidth usage', 'E.i4213_256to2 Fixed broadband, 256 kbit/s to less than 2 Mbit/s',
             'E.i4213_2to10 Fixed broadband, 2 to less than 10 Mbit/s', 'E.i4213_G10 Fixed broadband, equal to or above 10 Mbit/s ']

    wti_country = wti[(wti.ShortName == country_name)].pivot(index='Code', columns='DataYear', values = 'ShortName').reindex(columns=years, index=codes).notna().astype(int).reset_index()

    wti_country.Code=new_names

# Convert DataFrame to numpy array
    code_array = wti_country.set_index( 'Code').to_numpy()

    fig, ax = plt.subplots(figsize=(code_array.shape[1] * 1, code_array.shape[0] * 0.4))

# Show barcode
    ax.imshow(code_array, cmap='binary', aspect='auto', interpolation='nearest')

    ax.set_title(f'{country_name} Supply Side Data Availability by Year and Indicator', fontsize=14, pad=30)

# Set row labels (y-axis)
    ax.set_yticks(np.arange(code_array.shape[0]))
    ax.set_yticklabels(wti_country.set_index( 'Code').index)

# Set column labels (x-axis)
    ax.set_xticks(np.arange(code_array.shape[1]))
    ax.set_xticklabels(wti_country.set_index( 'Code').columns, rotation=0)
    ax.xaxis.tick_top() # put year lables on top

# Optional: remove grid / frame
    ax.set_xticks(np.arange(-.5, code_array.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-.5, code_array.shape[0], 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
    ax.tick_params(which='minor', bottom=False, left=False)

    plt.show()


    ## gauge
    year_columns = [2020, 2021, 2022, 2023, 2024]

    # Calculate your percentage
    completeness = wti_country[year_columns].sum().sum()/(14*5)*100

    # Create gauge chart
    fig, ax = plt.subplots(figsize=(8, 4), subplot_kw={'projection': 'polar'})

    # Set up the gauge (semicircle)
    theta = np.linspace(0, np.pi, 100)

    # Background arc (gray)
    ax.plot(theta, [1]*100, color='lightgray', linewidth=20)

    # Colored arc based on percentage
    # Define color zones
    if completeness >= 75:
        color = '#2ecc71'  # Green
    elif completeness >= 50:
        color = '#f39c12'  # Orange
    else:
        color = '#e74c3c'  # Red

    # Calculate how much of the arc to fill
    theta_fill = np.linspace(0, np.pi * (completeness/100), 100)
    ax.plot(theta_fill, [1]*100, color=color, linewidth=20)

    # Add percentage text in center
    ax.text(np.pi/2, 0.5, f'{completeness:.1f}%',
        ha='center', va='center', fontsize=20, fontweight='bold')
    # Add calculation text in center
    ax.text(np.pi/2, 0.3, f'{completeness*0.70:.0f}/70',
        ha='center', va='center', fontsize=20, fontweight='bold')
    ax.text(np.pi/2, 0.1, f'{country_name} Data Points',
        ha='center', va='center', fontsize=12, color='gray')

    # Customize appearance
    ax.set_ylim(0, 1)
    ax.set_theta_offset(np.pi)
    ax.set_theta_direction(-1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['polar'].set_visible(False)
    ax.grid(False)
    plt.subplots_adjust(bottom=0.1, top=0.9)

    plt.tight_layout()
    plt.show()


    ## line chart


    # Calculate sum for each year (excluding the 'Indicator' column)
    year_sums = wti_country[year_columns].sum()

    # Create line chart
    plt.figure(figsize=(8.2, 3))
    plt.plot(year_sums.index, year_sums.values, marker='o', linewidth=2, markersize=8)

    plt.title(f'{country_name} Number of Available Indicators by Year', fontsize=14, pad=20)
    # Set x-ticks to integers only
    plt.xticks(year_columns)  # This ensures only your year values appear as ticks
    #plt.xlabel('Year', fontsize=12)
    plt.ylabel('Number of Indicators', fontsize=12)
    #plt.grid(True, alpha=0.3)
    plt.ylim(0, max(year_sums.values) + 1)  # Add some space at the top
    plt.ylim(-1, max(year_sums.values) + 1)  # Starts at -1 instead of 0

    plt.tight_layout()
    plt.show()

    # Heatmap for the indicator group
    country_data = pivoted_group[pivoted_group['ShortName'] == country_name].drop('ShortName', axis=1)

    plt.figure(figsize=(12, 3))
    sns.heatmap(country_data[['A. Contracting_records_5Y', 'B. Cell_tower_coverage_5Y', 'C. Operator_traffic_5Y', 'D. Operators_wholesale_5Y', 'E. Contracting_fixed_broadband_speed_5Y']],
            annot=True,
            fmt='.0f',
            cmap='RdYlGn',  # Red-Yellow-Green
            cbar_kws={'label': 'Average Availability'},
            linewidths=1,
            linecolor='white',
            vmin=0,
            vmax=5)

    plt.title(f'{country_name} Availability by Indicator Group', fontsize=14, pad=20)
    #plt.xlabel('Indicator', fontsize=12)
    #plt.ylabel('')
    plt.yticks([])
    plt.xticks(rotation=45, ha='right')
    #plt.tight_layout()
    plt.show()
